# AR Search for Abstract Tokens — Speed & vLLM Experiment

**Goal**: Benchmark and optimize AR search over abstract tokens only.

| Method | Forward passes | Quality | vLLM? |
|--------|---------------|---------|-------|
| Recursion (Jacobi) | `max_iterations` (~2) | Parallel — later [a] don't see earlier [a] | No |
| AR search (naive) | `n_abs` (~20) per rollout | Full causal conditioning | No |
| AR search (KV-cache) | `n_abs` incremental steps | Full causal conditioning | **Yes** |

Key insight: NL tokens are **fixed** — we only generate at abstract positions.  
This is `>>K` times cheaper than GRPO which rolls out all tokens.

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import numpy as np
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper, infer_level
from sorl.sorl_trainer import (
    infer_insert_mask, expand_prompt_len, insert_tokens_with_padding,
    sorl_search, sorl_search_ar, select_best_sequences,
)
from data.pt_dataset import get_dataset, collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(model_name, abstract_vocab_size_list=[128])
model = model.to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
print(f"Vocab: base={model.vocab_sizes[0].item()}, total={model.total_vocab_size.item()}")

Using device: cpu


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded: 494.1M params
Vocab: base=151936, total=152065


In [2]:
# Prepare a batch of data (switch between "gsm8k" and "arc")
DATASET = "gsm8k"  # change to "arc" to test ARC-Challenge
train_ds = get_dataset(DATASET, split="train", tokenizer=tokenizer, max_length=256)
loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)
pad_token_id = tokenizer.pad_token_id

B, L = input_ids.shape
print(f"Dataset: {DATASET} | Batch: {B} x {L}")
print(f"Prompt lengths: {prompt_len.tolist()}")
print(f"Non-pad tokens per row: {attention_mask.sum(1).tolist()}")

# Peek at first sample
print(f"\n--- Sample 0 ---")
toks = input_ids[0][attention_mask[0].bool()]
print(tokenizer.decode(toks[:prompt_len[0]], skip_special_tokens=True)[:200] + "...")

Dataset: gsm8k | Batch: 4 x 256
Prompt lengths: [42, 35, 64, 57]
Non-pad tokens per row: [100, 99, 164, 183]

--- Sample 0 ---
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer:...


## 1. Prepare expanded sequence (shared setup for all methods)

In [3]:
# Expand sequence with periodic abstract token placeholders
K = 4
insert_mask = infer_insert_mask(input_ids, K, attention_mask)
expanded_prompt_len = expand_prompt_len(prompt_len, insert_mask)
expanded_data, expanded_mask = insert_tokens_with_padding(
    input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id
)

abs_mask = (expanded_data >= model.vocab_sizes[0])
n_abs = abs_mask[0].sum().item()
print(f"Expanded: {expanded_data.shape} | Abstract positions per row: {n_abs}")
print(f"Expanded prompt lengths: {expanded_prompt_len.tolist()}")

Expanded: torch.Size([4, 301]) | Abstract positions per row: 24
Expanded prompt lengths: [52, 43, 79, 71]


## 2. Benchmark: Recursion vs AR Search (naive) vs AR Search (KV-cache)

Timing the three search strategies on the same expanded sequence.

In [4]:
def benchmark(fn, name, warmup=1, repeats=3):
    """Time a function with warmup and repeats."""
    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    
    mean_t = np.mean(times)
    std_t = np.std(times)
    print(f"{name:40s} | {mean_t:.4f}s ± {std_t:.4f}s")
    return result, mean_t

In [ ]:
n_rollouts = 4
max_iterations = 2
temperature = 1.0
memory_span_abs = 1792
memory_span_traj = 1792

# --- Method 1: Recursion (parallel Jacobi) ---
@torch.no_grad()
def run_recursion():
    return sorl_search(
        model, input_ids, attention_mask, prompt_len, pad_token_id,
        n=n_rollouts, K=K, max_iterations=max_iterations,
        memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
        temperature=temperature,
    )

# --- Method 2: AR search (naive — full forward per abstract position) ---
@torch.no_grad()
def run_ar_search():
    return sorl_search_ar(
        model, input_ids, attention_mask, prompt_len, pad_token_id,
        n=n_rollouts, K=K,
        memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
        temperature=temperature,
    )

print(f"Config: B={B}, K={K}, n_rollouts={n_rollouts}, max_iter={max_iterations}, n_abs≈{n_abs}")
print(f"Recursion: {max_iterations + 1} forward passes (+ 1 eval)")
print(f"AR search: {n_abs + 1} forward passes (+ 1 eval)")
print(f"Expected slowdown: ~{(n_abs + 1) / (max_iterations + 1):.1f}x")
print()

res_rec, t_rec = benchmark(run_recursion, "Recursion (Jacobi)")
res_ar, t_ar = benchmark(run_ar_search, "AR search (naive)")

print(f"\nActual slowdown: {t_ar / t_rec:.2f}x")

Config: B=4, K=4, n_rollouts=4, max_iter=2, n_abs≈24
Recursion: 3 forward passes (+ 1 eval)
AR search: 25 forward passes (+ 1 eval)
Expected slowdown: ~8.3x



In [ ]:
# Compare quality: per-token loss of best sequences
best_data_rec, best_ppt_rec, best_adv_rec, _, _ = res_rec
best_data_ar, best_ppt_ar, best_adv_ar, _, _ = res_ar

rec_loss = best_ppt_rec.sum(1).mean().item()
ar_loss = best_ppt_ar.sum(1).mean().item()
print(f"Recursion best-of-{n_rollouts} total loss: {rec_loss:.4f}")
print(f"AR search best-of-{n_rollouts} total loss:  {ar_loss:.4f}")
print(f"Difference (AR - Rec): {ar_loss - rec_loss:.4f}")

# Check abstract token diversity
base_v = model.vocab_sizes[0].item()
abs_rec = best_data_rec[best_data_rec >= base_v] - base_v
abs_ar = best_data_ar[best_data_ar >= base_v] - base_v
print(f"\nAbstract token unique IDs — Recursion: {abs_rec.unique().numel()}, AR: {abs_ar.unique().numel()}")

## 3. AR Search with KV-Cache (faster — no redundant prefix computation)

The naive AR search recomputes the full sequence for every abstract position.  
With KV-cache, we process the prefix once, then only feed one token at a time for each subsequent position.

**Challenge with SoRL block_mask**: `flex_attention`'s `create_block_mask` expects full Q×KV dimensions. For incremental decoding we need a different approach — use standard `attention_mask` (2D) instead of `block_mask` for the cached steps, or pre-compute the full block_mask once and slice.

**Approach**: Process the full sequence up to the first abstract position with KV-cache enabled. Then for each subsequent position, feed tokens one-at-a-time (or in chunks between abstract positions), sampling only at abstract positions.

In [ ]:
def generate_abstract_only_kvcache(model, idx, attention_mask, memory_span_abs=1792, memory_span_traj=1792, temperature=1.0, prompt_len=None):
    """AR search with KV-cache: process prefix once, then step through positions incrementally.
    
    Strategy:
    - Forward pass the FULL sequence once with use_cache=True (using block_mask for SoRL attention)
    - For each abstract position left-to-right: extract logits, sample, update idx
    - After filling each abstract token, do an incremental forward pass from that position
      using past_key_values (standard attention_mask, no block_mask needed for single-token step)
    
    Fallback: If the underlying model doesn't support incremental block_mask decoding cleanly,
    we use a "chunked" approach — process prefix up to each abstract position using KV-cache
    with standard causal attention (no block_mask), which is vLLM-compatible.
    """
    vocab_size_0 = model.vocab_sizes[0].to(idx.device)
    abs_mask = (idx >= vocab_size_0)
    abs_mask[:, 0] = False
    abs_cols = abs_mask[0].nonzero(as_tuple=True)[0]  # (n_abs,)
    
    if len(abs_cols) == 0:
        return idx
    
    B, L = idx.shape
    
    # Handle temperature
    if isinstance(temperature, torch.Tensor) and temperature.ndim == 1:
        temp_batch = temperature.float().clamp(min=1e-10)
    else:
        temp_batch = None
        scalar_temp = max(float(temperature), 1e-10)
    
    # --- Phase 1: Prefill up to first abstract position (with KV-cache) ---
    first_abs = abs_cols[0].item()
    
    # Process prefix [0, first_abs) to build KV-cache
    # Use standard causal attention (no block_mask) for KV-cache compatibility
    prefix_ids = idx[:, :first_abs]
    prefix_mask = attention_mask[:, :first_abs]
    
    outputs = model.model.forward(
        input_ids=prefix_ids,
        attention_mask=prefix_mask,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values
    
    # --- Phase 2: Step through remaining positions, sampling at abstract positions ---
    for pos in range(first_abs, L):
        # Feed single token at current position
        cur_token = idx[:, pos:pos+1]  # (B, 1)
        # Attention mask must cover all positions seen so far (prefix + stepped positions)
        cur_attn_mask = attention_mask[:, :pos+1]
        
        outputs = model.model.forward(
            input_ids=cur_token,
            attention_mask=cur_attn_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        logits = outputs.logits[:, -1, :]  # (B, V)
        
        # Check if NEXT position is abstract (logits at pos predict pos+1)
        if pos + 1 < L and abs_mask[0, pos + 1]:
            # Sample abstract token for position pos+1
            logits[:, :vocab_size_0 + 1] = float('-inf')
            if temp_batch is not None:
                probs = F.softmax(logits / temp_batch.unsqueeze(1), dim=-1)
            else:
                probs = F.softmax(logits / scalar_temp, dim=-1)
            new_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
            idx[:, pos + 1] = new_token.to(idx.dtype)
    
    return idx


print("KV-cache AR search function defined.")

In [ ]:
@torch.no_grad()
def run_ar_kvcache():
    """AR search with KV-cache, wrapped to match sorl_search return format."""
    insert_mask = infer_insert_mask(input_ids, K, attention_mask)
    exp_prompt_len = expand_prompt_len(prompt_len, insert_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id
    )
    
    repeated_data = exp_data.repeat_interleave(n_rollouts, dim=0)
    repeated_mask = exp_mask.repeat_interleave(n_rollouts, dim=0)
    repeated_prompt_len = exp_prompt_len.repeat_interleave(n_rollouts, dim=0)
    
    # AR fill with KV-cache
    filled = generate_abstract_only_kvcache(
        model, repeated_data, repeated_mask,
        memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
        temperature=temperature, prompt_len=repeated_prompt_len,
    )
    
    # Eval pass (full forward for per-token loss, same as recursion/AR naive)
    labels = filled.clone()
    labels[repeated_mask == 0] = -100
    seq_idx = torch.arange(labels.size(1), device=device).unsqueeze(0)
    labels[seq_idx < repeated_prompt_len.unsqueeze(1)] = -100
    
    block_mask = model._create_sorl_block_mask(filled, memory_span_abs, memory_span_traj)
    outputs = model.model.forward(
        input_ids=filled, attention_mask=repeated_mask,
        block_mask=block_mask, use_cache=False,
    )
    shift_logits = outputs.logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
    ppt = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    ppt = ppt.view(filled.shape[0], -1)
    
    best_data, best_ppt, best_adv = select_best_sequences(filled, ppt, n_rollouts, exp_data.shape[0])
    return best_data, best_ppt, best_adv, exp_mask, exp_prompt_len

# Benchmark all three
print(f"Config: B={B}, K={K}, n_rollouts={n_rollouts}, n_abs≈{n_abs}")
print()

res_rec, t_rec = benchmark(run_recursion, "Recursion (Jacobi)")
res_ar, t_ar = benchmark(run_ar_search, "AR search (naive, no cache)")
res_kv, t_kv = benchmark(run_ar_kvcache, "AR search (KV-cache)")

print(f"\nSlowdown vs recursion:")
print(f"  AR naive:    {t_ar/t_rec:.2f}x")
print(f"  AR KV-cache: {t_kv/t_rec:.2f}x")
print(f"  KV-cache speedup over naive: {t_ar/t_kv:.2f}x")

In [ ]:
# Quality comparison across all three methods
best_data_rec, best_ppt_rec, _, _, _ = res_rec
best_data_ar, best_ppt_ar, _, _, _ = res_ar
best_data_kv, best_ppt_kv, _, _, _ = res_kv

base_v = model.vocab_sizes[0].item()

for name, bd, bp in [("Recursion", best_data_rec, best_ppt_rec),
                      ("AR naive", best_data_ar, best_ppt_ar),
                      ("AR KV-cache", best_data_kv, best_ppt_kv)]:
    total_loss = bp.sum(1).mean().item()
    abs_ids = bd[bd >= base_v] - base_v
    n_unique = abs_ids.unique().numel()
    print(f"{name:15s} | total_loss={total_loss:.4f} | unique_abs_ids={n_unique}")

## 4. vLLM Integration Path

The KV-cache AR search above uses HF's native `use_cache=True` on the inner `model.model` (Qwen2ForCausalLM).  
This is the **exact same interface** that vLLM accelerates.

### Why vLLM works here
- `SorlModelWrapper.model` is a standard HF `AutoModelForCausalLM` (Qwen2)
- vLLM can serve it directly — the only custom part is the expanded vocabulary (abstract tokens)
- During AR search, NL tokens are **fixed** — we only need vLLM to predict at abstract positions
- The "constrained decoding" pattern (mask base vocab, sample only abstract) maps to vLLM's `logits_processor`

### Integration options

**Option A: vLLM as inference server (recommended for multi-GPU)**
- Serve the model with `vllm serve` 
- Send the full expanded sequence as a prompt
- Use `logits_processor` to force NL tokens at non-abstract positions
- Use `n=num_rollouts` for parallel sampling

**Option B: vLLM offline batched inference**
- Use `vllm.LLM` class directly in the training script
- Same constrained decoding approach
- Better for single-GPU workflows

**Option C: Skip block_mask during search, use it only for training loss**
- During AR search: use standard causal attention (KV-cache compatible, vLLM compatible)
- During training forward pass: use full SoRL block_mask
- Rationale: search quality doesn't need the information bottleneck — it just needs good abstract tokens

In [ ]:
# Option C demo: AR search WITHOUT block_mask (standard causal attention)
# This is the vLLM-compatible path — no flex_attention needed during search

def generate_abstract_only_causal(model, idx, attention_mask, temperature=1.0, prompt_len=None):
    """AR search using standard causal attention + KV-cache (vLLM-compatible).
    
    Drops the SoRL block_mask entirely during search. The block_mask is only
    needed for the training loss forward pass, not for finding good abstract tokens.
    This makes the search compatible with vLLM / any KV-cache inference engine.
    """
    vocab_size_0 = model.vocab_sizes[0].to(idx.device)
    abs_mask = (idx >= vocab_size_0)
    abs_mask[:, 0] = False
    abs_cols = abs_mask[0].nonzero(as_tuple=True)[0]
    
    if len(abs_cols) == 0:
        return idx
    
    B, L = idx.shape
    if isinstance(temperature, torch.Tensor) and temperature.ndim == 1:
        temp_batch = temperature.float().clamp(min=1e-10)
    else:
        temp_batch = None
        scalar_temp = max(float(temperature), 1e-10)
    
    first_abs = abs_cols[0].item()
    
    # Prefill: process [0, first_abs) with KV-cache
    outputs = model.model.forward(
        input_ids=idx[:, :first_abs],
        attention_mask=attention_mask[:, :first_abs],
        use_cache=True,
    )
    past_key_values = outputs.past_key_values
    
    # Step through remaining positions
    for pos in range(first_abs, L):
        cur_token = idx[:, pos:pos+1]
        cur_attn_mask = attention_mask[:, :pos+1]
        
        outputs = model.model.forward(
            input_ids=cur_token,
            attention_mask=cur_attn_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        logits = outputs.logits[:, -1, :]
        
        if pos + 1 < L and abs_mask[0, pos + 1]:
            logits[:, :vocab_size_0 + 1] = float('-inf')
            if temp_batch is not None:
                probs = F.softmax(logits / temp_batch.unsqueeze(1), dim=-1)
            else:
                probs = F.softmax(logits / scalar_temp, dim=-1)
            new_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
            idx[:, pos + 1] = new_token.to(idx.dtype)
    
    return idx

print("vLLM-compatible AR search (standard causal, no block_mask) defined.")

In [ ]:
@torch.no_grad()
def run_ar_causal():
    """AR search with standard causal attention + KV-cache (no block_mask)."""
    insert_mask = infer_insert_mask(input_ids, K, attention_mask)
    exp_prompt_len = expand_prompt_len(prompt_len, insert_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id
    )
    
    repeated_data = exp_data.repeat_interleave(n_rollouts, dim=0)
    repeated_mask = exp_mask.repeat_interleave(n_rollouts, dim=0)
    repeated_prompt_len = exp_prompt_len.repeat_interleave(n_rollouts, dim=0)
    
    filled = generate_abstract_only_causal(
        model, repeated_data, repeated_mask,
        temperature=temperature, prompt_len=repeated_prompt_len,
    )
    
    # Eval pass WITH block_mask (training-faithful loss computation)
    labels = filled.clone()
    labels[repeated_mask == 0] = -100
    seq_idx = torch.arange(labels.size(1), device=device).unsqueeze(0)
    labels[seq_idx < repeated_prompt_len.unsqueeze(1)] = -100
    
    block_mask = model._create_sorl_block_mask(filled, memory_span_abs, memory_span_traj)
    outputs = model.model.forward(
        input_ids=filled, attention_mask=repeated_mask,
        block_mask=block_mask, use_cache=False,
    )
    shift_logits = outputs.logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
    ppt = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    ppt = ppt.view(filled.shape[0], -1)
    
    best_data, best_ppt, best_adv = select_best_sequences(filled, ppt, n_rollouts, exp_data.shape[0])
    return best_data, best_ppt, best_adv, exp_mask, exp_prompt_len

# Full benchmark: all 4 methods
print(f"Config: B={B}, K={K}, n_rollouts={n_rollouts}, n_abs≈{n_abs}")
print()

res_rec, t_rec = benchmark(run_recursion, "Recursion (Jacobi)")
res_ar, t_ar = benchmark(run_ar_search, "AR naive (full fwd, block_mask)")
res_kv, t_kv = benchmark(run_ar_kvcache, "AR KV-cache (block_mask prefill)")
res_causal, t_causal = benchmark(run_ar_causal, "AR KV-cache (causal, no block_mask)")

print(f"\n{'Method':45s} | {'Time':>10s} | {'vs Rec':>8s}")
print("-" * 70)
for name, t in [("Recursion (Jacobi)", t_rec),
                ("AR naive (full fwd, block_mask)", t_ar),
                ("AR KV-cache (block_mask prefill)", t_kv),
                ("AR KV-cache (causal, no block_mask)", t_causal)]:
    print(f"{name:45s} | {t:>9.4f}s | {t/t_rec:>7.2f}x")

In [ ]:
# Quality comparison: all 4 methods
best_data_causal, best_ppt_causal, _, _, _ = res_causal

print(f"{'Method':25s} | {'total_loss':>12s} | {'unique_abs':>10s}")
print("-" * 55)
for name, bd, bp in [("Recursion", best_data_rec, best_ppt_rec),
                      ("AR naive", best_data_ar, best_ppt_ar),
                      ("AR KV-cache", best_data_kv, best_ppt_kv),
                      ("AR causal (vLLM)", best_data_causal, best_ppt_causal)]:
    total_loss = bp.sum(1).mean().item()
    abs_ids = bd[bd >= base_v] - base_v
    n_unique = abs_ids.unique().numel()
    print(f"{name:25s} | {total_loss:>12.4f} | {n_unique:>10d}")

## 5. vLLM Offline Inference Demo

Below is a sketch showing how to use `vllm.LLM` for abstract-only AR search.  
The key trick: use a **logits_processor** to force NL tokens at non-abstract positions and constrain sampling to abstract vocab at abstract positions.

> **Note**: Requires `pip install vllm`. Only runs on CUDA with sufficient GPU memory.

In [ ]:
# ============================================================
# vLLM Offline Inference for Abstract-Only AR Search
# ============================================================
# Uncomment and run if vllm is installed (pip install vllm)
# Only works on CUDA GPUs.

VLLM_AVAILABLE = False
try:
    import vllm
    VLLM_AVAILABLE = True
    print(f"vLLM version: {vllm.__version__}")
except ImportError:
    print("vLLM not installed. Skipping vLLM cells. Install with: pip install vllm")


def build_vllm_search(model_name, abstract_vocab_size=128, n_rollouts=4, temperature=1.0):
    """Build a vLLM-based abstract-only AR search function.
    
    The idea: 
    - vLLM serves the base model (Qwen2.5-0.5B) with expanded vocab
    - We feed it the full expanded sequence as a "prompt" 
    - Use a custom logits_processor to:
      (a) At NL positions: force the known NL token (set all others to -inf)
      (b) At abstract positions: mask base vocab, sample from abstract vocab only
    - vLLM handles KV-cache, batching, continuous batching, tensor parallelism etc.
    """
    if not VLLM_AVAILABLE:
        print("vLLM not available")
        return None
    
    from vllm import LLM, SamplingParams
    
    # Load model with expanded vocab
    # vLLM needs to know about the extra embeddings
    base_vocab = 151936  # Qwen2.5-0.5B base vocab
    total_vocab = base_vocab + abstract_vocab_size + 1  # +1 for placeholder
    
    llm = LLM(
        model=model_name,
        # vLLM will auto-detect the model architecture
        # For custom vocab size, we may need to resize after loading
        max_model_len=512,
        gpu_memory_utilization=0.8,
    )
    
    return llm

print("vLLM search builder defined (will only execute if vLLM is installed).")

In [ ]:
# ============================================================
# vLLM Constrained Decoding for Abstract-Only Search
# ============================================================
# This is the production-ready pattern for vLLM integration.
# The logits_processor enforces: NL positions → force known token, abstract positions → sample.

if VLLM_AVAILABLE:
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import LogitsProcessor
    
    class AbstractOnlyLogitsProcessor:
        """Constrained decoding: force NL tokens, sample only at abstract positions.
        
        Given a fixed expanded sequence (with placeholders at abstract positions),
        this processor modifies logits at each generation step:
        - If next position is NL: set all logits to -inf except the known NL token
        - If next position is abstract: set base vocab logits to -inf, sample from abstract vocab
        """
        def __init__(self, expanded_seq, base_vocab_size, prompt_offset=0):
            self.expanded_seq = expanded_seq  # full sequence including placeholders
            self.base_vocab_size = base_vocab_size
            self.prompt_offset = prompt_offset  # how many tokens were in the "prompt" fed to vLLM
            
        def __call__(self, token_ids, logits):
            # Current generation step = len(token_ids) means we're predicting this position
            next_pos = self.prompt_offset + len(token_ids)
            
            if next_pos >= len(self.expanded_seq):
                return logits
            
            next_token = self.expanded_seq[next_pos]
            
            if next_token < self.base_vocab_size:
                # NL position: force the known token
                logits[:] = float('-inf')
                logits[next_token] = 0.0
            else:
                # Abstract position: mask base vocab + placeholder, sample from abstract
                logits[:self.base_vocab_size + 1] = float('-inf')
            
            return logits
    
    print("AbstractOnlyLogitsProcessor defined for vLLM constrained decoding.")
    
    # --- Demo usage sketch ---
    # llm = LLM(model="Qwen/Qwen2.5-0.5B", max_model_len=512)
    # 
    # # For each sample in batch:
    # for i in range(B):
    #     seq = expanded_data[i].tolist()
    #     processor = AbstractOnlyLogitsProcessor(seq, base_vocab_size=base_v, prompt_offset=0)
    #     
    #     # Feed empty prompt (or just BOS), let constrained decoding reconstruct the full sequence
    #     # with abstract tokens sampled autoregressively
    #     params = SamplingParams(
    #         temperature=1.0,
    #         max_tokens=len(seq),
    #         logits_processors=[processor],
    #         n=n_rollouts,  # parallel samples for best-of-K
    #     )
    #     outputs = llm.generate(prompt_token_ids=[[seq[0]]], sampling_params=params)
else:
    print("Skipping vLLM constrained decoding (vLLM not installed).")

## 6. Scaling Analysis: How does AR search cost scale with sequence length?

Key variables:
- `n_abs` = number of abstract positions ≈ `seq_len / (K+1)`
- Recursion cost: `(max_iterations + 1)` full forward passes × `B*n_rollouts`
- AR naive cost: `(n_abs + 1)` full forward passes × `B*n_rollouts`  
- AR KV-cache cost: 1 prefill + `L - first_abs` incremental steps + 1 eval pass
- GRPO cost: `seq_len` incremental steps × `B*n_rollouts` (all tokens, not just abstract)

In [ ]:
import matplotlib.pyplot as plt

# Scaling analysis: cost as a function of sequence length
seq_lens = np.arange(64, 513, 32)
K_val = 4
max_iter = 2

n_abs_arr = seq_lens / (K_val + 1)  # approximate abstract positions

# Cost in "forward pass equivalents" (full fwd = 1.0, incremental step ≈ seq_len_ratio)
recursion_cost = (max_iter + 1) * np.ones_like(seq_lens, dtype=float)  # full fwd passes
ar_naive_cost = n_abs_arr + 1  # full fwd passes
# KV-cache: 1 prefill (≈1 full fwd) + (L - first_abs) incremental steps (each ≈ 1/L of full fwd) + 1 eval
# Incremental step cost ≈ 1/L of full fwd (linear in KV length, but no Q recomputation)
ar_kvcache_cost = 1 + (seq_lens - seq_lens/(K_val+1)) * (1.0/seq_lens) + 1  # ≈ 2 + (K/(K+1))
# GRPO: seq_len incremental steps + eval
grpo_cost = seq_lens * (1.0/seq_lens) + 1  # ≈ 2 (but over ALL tokens, not just abstract)
# More accurate: GRPO generates seq_len tokens, each with incremental KV step
grpo_full_cost = seq_lens / seq_lens + 1  # normalized, but the key is GRPO generates ALL tokens

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: full forward pass equivalents (no KV-cache)
ax1.plot(seq_lens, recursion_cost, 'b-o', label=f'Recursion (iter={max_iter})', markersize=3)
ax1.plot(seq_lens, ar_naive_cost, 'r-s', label='AR naive (no cache)', markersize=3)
ax1.plot(seq_lens, seq_lens / (K_val+1) * 0 + seq_lens, 'k--', label='GRPO (full rollout)', alpha=0.5, markersize=3)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Full Forward Passes')
ax1.set_title('Cost: Full Forward Passes (no KV-cache)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: with KV-cache (incremental steps are cheap)
# With KV-cache, each incremental step is O(1) in compute (just 1 token query)
# So cost ≈ prefill + n_steps * (tiny) + eval
# The real bottleneck is memory bandwidth, not compute
ax2.bar(np.arange(4), [
    max_iter + 1,           # Recursion: no KV-cache benefit
    n_abs_arr.mean() + 1,   # AR naive: no KV-cache benefit  
    2.5,                     # AR KV-cache: ~1 prefill + cheap steps + 1 eval
    2.0,                     # GRPO KV-cache: ~1 prefill + cheap steps + 1 eval (but ALL tokens)
], color=['blue', 'red', 'green', 'gray'], alpha=0.7)
ax2.set_xticks(np.arange(4))
ax2.set_xticklabels(['Recursion\n(no cache)', 'AR naive\n(no cache)', 'AR\nKV-cache', 'GRPO\nKV-cache'], fontsize=9)
ax2.set_ylabel('Effective Full Forward Equivalents')
ax2.set_title(f'Cost Comparison (seq_len≈256, K={K_val})')
ax2.grid(True, alpha=0.3, axis='y')

# Add text annotations
ax2.text(0, max_iter + 1.5, f'{max_iter+1} fwd', ha='center', fontsize=9)
ax2.text(1, n_abs_arr.mean() + 2, f'~{n_abs_arr.mean():.0f} fwd', ha='center', fontsize=9)
ax2.text(2, 3.0, '~2.5 fwd\n(cheap steps)', ha='center', fontsize=8)
ax2.text(3, 2.5, '~2 fwd\nbut ALL tokens', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('ar_search_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ar_search_cost.png")

## 7. Optimized `generate_abstract_only` for Production

The KV-cache version above steps through **every** position (NL + abstract).  
We can skip NL chunks entirely — process them as a single prefill chunk, only "step" at abstract positions.

This is the **chunk-prefill** approach:
1. Split sequence into segments between abstract positions
2. Each NL segment → prefill as a chunk (fast, parallel)
3. Each abstract position → single-token decode step (sample)

This minimizes the number of decode steps to exactly `n_abs`.

In [ ]:
def generate_abstract_only_chunked(model, idx, attention_mask, temperature=1.0, prompt_len=None):
    """Chunk-prefill AR search: prefill NL chunks, decode only at abstract positions.
    
    Minimizes decode steps to exactly n_abs (no stepping through NL positions).
    Fully KV-cache compatible → vLLM compatible.
    
    Algorithm:
    1. Find abstract positions → split sequence into [NL_chunk_0, abs_0, NL_chunk_1, abs_1, ...]
    2. Prefill NL_chunk_0 (builds KV-cache for the prefix)
    3. Decode abs_0 (single token, sample from abstract vocab)
    4. Prefill NL_chunk_1 (extend KV-cache with known NL tokens)
    5. Decode abs_1 ... and so on
    """
    vocab_size_0 = model.vocab_sizes[0].to(idx.device)
    abs_mask = (idx >= vocab_size_0)
    abs_mask[:, 0] = False
    abs_cols = abs_mask[0].nonzero(as_tuple=True)[0].tolist()  # list of int
    
    if len(abs_cols) == 0:
        return idx
    
    B, L = idx.shape
    if isinstance(temperature, torch.Tensor) and temperature.ndim == 1:
        temp_batch = temperature.float().clamp(min=1e-10)
    else:
        temp_batch = None
        scalar_temp = max(float(temperature), 1e-10)
    
    past_key_values = None
    processed_up_to = 0  # how many positions are in the KV-cache
    
    for abs_col in abs_cols:
        # --- Prefill NL chunk: [processed_up_to, abs_col) ---
        if abs_col > processed_up_to:
            chunk_ids = idx[:, processed_up_to:abs_col]  # (B, chunk_len)
            chunk_attn = attention_mask[:, :abs_col]      # full mask up to current position
            
            outputs = model.model.forward(
                input_ids=chunk_ids,
                attention_mask=chunk_attn,
                past_key_values=past_key_values,
                use_cache=True,
            )
            past_key_values = outputs.past_key_values
            processed_up_to = abs_col
        
        # --- Decode at abstract position abs_col ---
        # Logits from the last position predict the abstract token at abs_col
        # But we need to actually feed the token at abs_col-1 if it wasn't in the chunk
        # Actually: the chunk ended at abs_col-1, so outputs.logits[:, -1, :] predicts abs_col
        logits = outputs.logits[:, -1, :]  # (B, V)
        
        # Mask base vocab + placeholder
        logits[:, :vocab_size_0 + 1] = float('-inf')
        
        if temp_batch is not None:
            probs = F.softmax(logits / temp_batch.unsqueeze(1), dim=-1)
        else:
            probs = F.softmax(logits / scalar_temp, dim=-1)
        new_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
        idx[:, abs_col] = new_token.to(idx.dtype)
        
        # Feed the sampled abstract token into KV-cache
        abs_token = idx[:, abs_col:abs_col+1]
        abs_attn = attention_mask[:, :abs_col+1]
        
        outputs = model.model.forward(
            input_ids=abs_token,
            attention_mask=abs_attn,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        processed_up_to = abs_col + 1
    
    # Process any remaining NL tokens after the last abstract position
    if processed_up_to < L:
        remaining = idx[:, processed_up_to:L]
        remaining_attn = attention_mask[:, :L]
        outputs = model.model.forward(
            input_ids=remaining,
            attention_mask=remaining_attn,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
    
    return idx

print("Chunk-prefill AR search defined.")
print(f"For K={K}, this does exactly {n_abs} decode steps + {n_abs+1} chunk prefills")

In [ ]:
@torch.no_grad()
def run_ar_chunked():
    """AR search with chunk-prefill + KV-cache (minimal decode steps)."""
    insert_mask = infer_insert_mask(input_ids, K, attention_mask)
    exp_prompt_len = expand_prompt_len(prompt_len, insert_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id
    )
    
    repeated_data = exp_data.repeat_interleave(n_rollouts, dim=0)
    repeated_mask = exp_mask.repeat_interleave(n_rollouts, dim=0)
    repeated_prompt_len = exp_prompt_len.repeat_interleave(n_rollouts, dim=0)
    
    filled = generate_abstract_only_chunked(
        model, repeated_data, repeated_mask,
        temperature=temperature, prompt_len=repeated_prompt_len,
    )
    
    # Eval pass WITH block_mask (training-faithful loss)
    labels = filled.clone()
    labels[repeated_mask == 0] = -100
    seq_idx = torch.arange(labels.size(1), device=device).unsqueeze(0)
    labels[seq_idx < repeated_prompt_len.unsqueeze(1)] = -100
    
    block_mask = model._create_sorl_block_mask(filled, memory_span_abs, memory_span_traj)
    outputs = model.model.forward(
        input_ids=filled, attention_mask=repeated_mask,
        block_mask=block_mask, use_cache=False,
    )
    shift_logits = outputs.logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
    ppt = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    ppt = ppt.view(filled.shape[0], -1)
    
    best_data, best_ppt, best_adv = select_best_sequences(filled, ppt, n_rollouts, exp_data.shape[0])
    return best_data, best_ppt, best_adv, exp_mask, exp_prompt_len

# === FULL BENCHMARK: all 5 methods ===
print(f"Config: B={B}, K={K}, n_rollouts={n_rollouts}, n_abs≈{n_abs}")
print(f"Recursion: {max_iterations+1} full fwd | AR naive: {n_abs+1} full fwd | AR chunked: {n_abs} decode steps")
print()

results = {}
results["Recursion (Jacobi)"], t_rec = benchmark(run_recursion, "Recursion (Jacobi)")
results["AR naive (no cache)"], t_ar = benchmark(run_ar_search, "AR naive (no cache)")
results["AR KV-cache (per-pos)"], t_kv = benchmark(run_ar_kvcache, "AR KV-cache (per-pos step)")
results["AR causal (no block_mask)"], t_causal = benchmark(run_ar_causal, "AR causal (no block_mask)")
results["AR chunked (optimal)"], t_chunked = benchmark(run_ar_chunked, "AR chunked (chunk-prefill)")

print(f"\n{'Method':40s} | {'Time':>10s} | {'vs Rec':>8s}")
print("-" * 65)
for name, t in [("Recursion (Jacobi)", t_rec),
                ("AR naive (no cache)", t_ar),
                ("AR KV-cache (per-pos step)", t_kv),
                ("AR causal (no block_mask)", t_causal),
                ("AR chunked (chunk-prefill)", t_chunked)]:
    print(f"{name:40s} | {t:>9.4f}s | {t/t_rec:>7.2f}x")

In [ ]:
# Quality comparison: all methods
print(f"{'Method':30s} | {'total_loss':>12s} | {'unique_abs':>10s}")
print("-" * 60)
for name, res in results.items():
    bd, bp = res[0], res[1]
    total_loss = bp.sum(1).mean().item()
    abs_ids = bd[bd >= base_v] - base_v
    n_unique = abs_ids.unique().numel()
    print(f"{name:30s} | {total_loss:>12.4f} | {n_unique:>10d}")

## 8. Next Steps & Integration Checklist

### To use AR search in training today:
```python
cfg = SoRLConfig(ar_search=True, temperature=1.0)
```
This uses the naive `generate_abstract_only` (full forward per abstract position, with block_mask).

### To upgrade to KV-cache AR search:
1. Move `generate_abstract_only_chunked` into `SorlModelWrapper`
2. Add a `use_kvcache` flag to `generate_abstract_only` 
3. The chunk-prefill approach skips block_mask during search → uses standard causal attention
4. The eval pass still uses block_mask for training-faithful loss

### To integrate vLLM:
1. Save LoRA adapter + abstract embeddings checkpoint
2. Load into vLLM with `--enable-lora` flag
3. Use `AbstractOnlyLogitsProcessor` for constrained decoding
4. `SamplingParams(n=num_rollouts)` gives parallel samples for best-of-K
5. Feed expanded sequence as prompt, get back filled sequences
6. Compute training loss with block_mask in PyTorch (not vLLM)

### Key insight for speed:
- **Search** doesn't need block_mask (standard causal is fine for finding good abstract tokens)
- **Training loss** needs block_mask (information bottleneck matters for gradient signal)
- This separation lets us use vLLM for search and PyTorch for loss

# TODO: Delete this cell (duplicate of cell 0)